In [1]:
import eqsql
import eqsql.task_queues
import eqsql.task_queues.local_queue
import json
import psycopg2

db_host = 'midway3-login3.rcc.local'
db_port = 54311
db_name = 'EQ_SQL'
db_user = 'eqsql_user'


In [5]:
from eqsql import db_tools

db_path = '/project/jozik/ncollier/db/varmodel'
bin_path = '/project/jozik/sfw/gcc-10.2.0/postgres-14.12/bin'
# db_tools.start_db(db_path, bin_path, db_port=54311)
db_tools.stop_db(db_path, bin_path, db_port)

Checking for pg_ctl ...
/project/jozik/sfw/gcc-10.2.0/postgres-14.12/bin/pg_ctl


Stopping database server

waiting for server to shut down.... done
server stopped



In [8]:
# Create UPF
# where json_in is null and task_type = X, write the json_out to a sweep file with the eq_task_id.
date = "11032025"
version="5.0"

# [400, 401, 402, 403, 500, 501, 502, 503]
conn = psycopg2.connect(f'dbname={db_name}', user=db_user, host=db_host, port=db_port)
with conn:
    with conn.cursor() as cur:
        for task_type in [401, 402, 403]:
            with open(f"../data/upfs/{task_type}_{date}_{version}_upf.txt", "w") as fout:
                sql = f"select eq_task_id, json_out from eq_tasks where json_in is null and eq_task_type = {task_type}"
                cur.execute(sql)

                for task_id, json_out in cur.fetchall():
                    params = json.loads(json_out)
                    params["task_id"] = task_id
                    fout.write(f"{json.dumps(params)}\n")

In [6]:
# Find finished runs in UPF sweep
import glob
import json
import os

task_type = 403
oo = "off"
root = f"/project/jozik/ncollier/repos/varmodel3/emews/experiments/gi_{oo}_032025_lhs_sweep_{task_type}_4.0/instances"
fs = glob.glob(f"{root}/*/err.txt")

def read_params(instance_dir):
    with open(os.path.join(instance_dir, "parameters.json")) as fin:
        params = json.load(fin)
        r = {"output_sqlite": os.path.join(instance_dir, "output.sqlite"),
             "instance_dir": instance_dir,
             "task_id": params["task_id"],
             "task_type": task_type}
        return r

all_params = []

for f in fs:
    with open(f) as fin:
        for line in fin.readlines():
            if "elapsed time" in line:
                all_params.append(read_params(os.path.dirname(f)))
                found = True
                break

           

len(all_params)


359

In [7]:
# Add resutls to DB
    
task_queue = eqsql.task_queues.local_queue.init_task_queue(db_host, db_user, db_port,
                                                           db_name)

for p in all_params:
    eq_task_id = p["task_id"]
    task_type = p["task_type"]
    output = p["output_sqlite"]
    task_queue.report_task(eq_task_id, task_type, output)

task_queue.close()

In [8]:
# get runtimes, mem for all completed runs
# elapsed time (s): 41303.57
# maxrss (GB) = 4.929203033447266
# in err.txt
import json
import os

conn = psycopg2.connect(f'dbname={db_name}', user=db_user, host=db_host, port=db_port)
with conn:
    with conn.cursor() as cur:
        sql = "select json_in, json_out, eq_task_type from eq_tasks where json_in is not null and time_created > '2025-04-02' and eq_task_type in (400, 401, 402, 403, 500, 501, 502, 503)"
        cur.execute(sql)
        metrics = []
        for json_in, json_out, task_type in cur.fetchall():
            params = json.loads(json_out)
            metric = [params, -1.0, -1.0, "on" if task_type > 499 else "off"]
            metrics.append(metric)
            instance_dir = os.path.dirname(json_in)
            with open(os.path.join(instance_dir, "err.txt")) as fin:
                for line in fin.readlines():
                    line = line.strip()
                    if line.startswith("elapsed time"):
                        idx = line.find(":")
                        runtime = float(line[idx + 1:].strip())
                        metric[1] = runtime
                    elif line.startswith("maxrss"):
                        idx = line.find("=")
                        mem = float(line[idx + 1:].strip())
                        metric[2] = mem

len(metrics)



15357

In [9]:
import csv

header = [x for x in metrics[0][0].keys()]

with open("gi_032025_lhs_metrics.csv", "w") as fout:
    writer = csv.writer(fout)
    writer.writerow(header + ["gi", "runtime", "memory"])
    for params, rt, mem, gi in metrics:
        writer.writerow([params[h] for h in header] + [gi, rt, mem])

In [4]:
## GET ALL RESULTS FOR ON / OFF ##

import sqlite3
import csv

def get_target_names(f):
    with sqlite3.connect(f) as con:
        cur = con.execute(f'select * from targets')
        targets = [description[0] for description in cur.description]
        return targets
    
def get_targets(f, instance):
    with sqlite3.connect(f) as con:
        cur = con.execute("select * from targets")
        return [[instance] + list(vals) for vals in cur.fetchall()]
# Gather Results from sqlite output
conn = psycopg2.connect(f'dbname={db_name}', user=db_user, host=db_host, port=db_port)
# tt_list = [str(x) for x in [500, 501, 502, 503]]
tt_list = [str(x) for x in [400, 401, 402, 403]]
sql_task_types = ",".join(tt_list)

with conn:
    with conn.cursor() as cur:
        cur.execute(f'select json_in, json_out from eq_tasks where eq_task_type in ({sql_task_types}) and eq_status = 2')
        json_in, json_out = cur.fetchone()    
        target_names = get_target_names(json_in)
        param_names = [k for k, _ in json.loads(json_out).items()]

sql_targets = ",".join(target_names)
sql_targets

all_targets = []
all_params = []
with conn:
    with conn.cursor() as cur:
        cur.execute(f'select json_in, json_out from eq_tasks where eq_task_type in ({sql_task_types}) and eq_status = 2')
        for json_in, json_out in cur.fetchall():
            params = json.loads(json_out)
            all_params.append([v for _, v in params.items()])
            instance = params['instance']
            targets = get_targets(json_in, instance)
            all_targets += targets

run_type = "off"
date_infix = "11212025"
f = f"gi_{run_type}_results_{date_infix}.csv"
with open(f, 'w') as fout:
    writer = csv.writer(fout)
    writer.writerow(["instance"] + target_names)
    writer.writerows(all_targets)

f = f"gi_{run_type}_parameters_{date_infix}.csv"
with open(f, "w") as fout:
    writer = csv.writer(fout)
    writer.writerow(param_names)
    writer.writerows(all_params)

In [13]:
# Get run times for each completed instance
import glob
import json
import os

def get_elapsed_time(sqlfile):
    dn = os.path.dirname(sqlfile)
    f = os.path.join(dn, "err.txt")
    with open(f) as fin:
        for line in fin.readlines():
            if "elapsed time" in line:
                elapsed_time = float(line.split(' ')[3])
                return elapsed_time

runtimes = {}

conn = psycopg2.connect(f'dbname={db_name}', user=db_user, host=db_host, port=db_port)
with conn:
    with conn.cursor() as cur:
        sql = """
            select json_out, json_in 
            from eq_tasks where json_in is not null and eq_task_type in (500, 501, 502, 503) and
            time_created > '04-01-2025'
        """
        cur.execute(sql)
        for json_out, json_in in cur.fetchall():
            params = json.loads(json_out)
            instance = params["instance"]
            elapsed_time = get_elapsed_time(json_in)
            runtimes[instance] = elapsed_time / 60

len(runtimes)



6954

In [14]:
import glob
import pandas as pd
import os

def find_runtime(x, runtimes):
    return runtimes[x["instance"]]

fs = glob.glob("midwayAllocationRequest/gi_on*")
for f in fs:
    df = pd.read_csv(f)
    df["runtime"] = df.apply(find_runtime, axis=1, args=(runtimes,))
    nf = os.path.basename(f)
    nf = f"./runtime_results/{os.path.splitext(nf)[0]}_runtime.csv"
    df.to_csv(nf)


In [9]:
import glob
import os
import pandas as pd
from io import StringIO

# buf = StringIO()
fs = glob.glob("./runtime_results/*")
lines = []
for f in fs:
    df = pd.read_csv(f)
    d = df.runtime.describe().to_dict()
    d["fname"] = os.path.basename(f)
    lines.append(d)

df = pd.DataFrame.from_records(lines)
df


,count,mean,std,min,25%,50%,75%,max,fname
0,39.0,280.464348,89.444793,159.223967,227.456742,262.168217,308.093133,614.110367,gi_on_threshold_78_PCR_sensitivity_level_1_run...
1,575.0,445.939050,153.944013,181.423833,339.335025,423.209367,522.813542,1086.367950,gi_off_threshold_84_PCR_sensitivity_level_1_ru...
2,699.0,546.601186,217.250514,186.685767,386.793992,502.135333,660.566167,1356.876250,gi_off_threshold_90_PCR_sensitivity_level_1_ru...
3,125.0,409.077109,129.368050,203.577933,322.829700,400.540833,458.101933,892.403833,gi_on_threshold_96_PCR_sensitivity_level_1_run...
4,364.0,357.466960,106.702200,163.284433,277.338921,344.780642,419.192833,753.772283,gi_off_threshold_78_PCR_sensitivity_level_1_ru...
5,117.0,398.290608,119.711751,162.882050,304.533967,394.470933,448.074267,892.403833,gi_on_threshold_90_PCR_sensitivity_level_1_run...
6,83.0,341.170049,102.588312,163.248617,270.135942,313.753600,408.977075,676.424167,gi_on_threshold_84_PCR_sensitivity_level_1_run...
7,772.0,586.586235,240.485139,186.685767,404.545254,551.493450,697.556567,1816.966850,gi_off_threshold_96_PCR_sensitivity_level_1_ru...


In [ ]:
# Get all completed results
# maxrss (GB) = 4.929203033447266
# in err.txt
import json
import os

conn = psycopg2.connect(f'dbname={db_name}', user=db_user, host=db_host, port=db_port)
with conn:
    with conn.cursor() as cur:
        sql = "select json_in, json_out, eq_task_type from eq_tasks where json_in is not null and time_created > '2025-04-02' and eq_task_type in (400, 401, 402, 403, 500, 501, 502, 503)"
        cur.execute(sql)
        metrics = []
        for json_in, json_out, task_type in cur.fetchall():
            params = json.loads(json_out)
            metric = [params, -1.0, -1.0, "on" if task_type > 499 else "off"]
            metrics.append(metric)
            instance_dir = os.path.dirname(json_in)
            with open(os.path.join(instance_dir, "err.txt")) as fin:
                for line in fin.readlines():
                    line = line.strip()
                    if line.startswith("elapsed time"):
                        idx = line.find(":")
                        runtime = float(line[idx + 1:].strip())
                        metric[1] = runtime
                    elif line.startswith("maxrss"):
                        idx = line.find("=")
                        mem = float(line[idx + 1:].strip())
                        metric[2] = mem

len(metrics)